In [10]:
## 3.Data cleaning and feature preparation.

## Check data quality.
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
df = pd.read_csv(r'C:\Users\Soumya_SRB\Desktop\L&T Project Intelligencee\India_Construction_Data_L&T.csv')


# Check the overall dataset
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nColumn data types:")
print(df.dtypes)

Dataset shape: (1000, 15)

Missing values:
Project_ID                  0
Project_Type                0
Region                      0
Planned_Duration_Months     0
Project_Progress_Percent    0
Budget_Million_DZD          0
Cost_Used_Percent           0
Number_of_Workers           0
Material_Availability       0
Equipment_Availability      0
Weather_Risk                0
Supplier_Delay              0
Project_Complexity          0
Safety_Incidents            0
Delay_Risk                  0
dtype: int64

Duplicate rows: 0

Column data types:
Project_ID                      str
Project_Type                    str
Region                          str
Planned_Duration_Months       int64
Project_Progress_Percent      int64
Budget_Million_DZD          float64
Cost_Used_Percent           float64
Number_of_Workers             int64
Material_Availability           str
Equipment_Availability          str
Weather_Risk                    str
Supplier_Delay                  str
Project_Complexity     

In [11]:
## Check numerical ranges.
display(df.describe().T)

# Check percentage columns
print("Progress outside 0–100:")
print(((df["Project_Progress_Percent"] < 0) |
       (df["Project_Progress_Percent"] > 100)).sum())

print("\nCost percentage below zero:")
print((df["Cost_Used_Percent"] < 0).sum())

,count,mean,std,min,25%,50%,75%,max
Planned_Duration_Months,1000.0,27.08300,12.222172,6.00,17.0000,27.000,38.0000,48.00
Project_Progress_Percent,1000.0,52.71000,27.441300,5.00,29.0000,53.000,76.0000,100.00
Budget_Million_DZD,1000.0,2537.09240,1409.982201,56.66,1332.5375,2576.255,3769.9675,4988.23
Cost_Used_Percent,1000.0,69.49486,34.520953,10.02,40.2875,69.505,99.4075,129.63
Number_of_Workers,1000.0,402.15200,222.329430,20.00,213.0000,400.000,595.0000,799.00
Safety_Incidents,1000.0,1.19700,1.096080,0.00,0.0000,1.000,2.0000,7.00


Progress outside 0–100:
0

Cost percentage below zero:
0


In [12]:
## Remove duplicates, if any.
before = len(df)
df = df.drop_duplicates()
after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 1000
Rows after: 1000
Duplicates removed: 0


In [13]:
## Check categorical values.
categorical_columns = [
    "Project_Type",
    "Region",
    "Material_Availability",
    "Equipment_Availability",
    "Weather_Risk",
    "Supplier_Delay",
    "Project_Complexity",
    "Delay_Risk"]

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))


Project_Type
Project_Type
Road               238
Building           200
Housing            172
Pipeline           169
Bridge             121
Industrial Site    100
Name: count, dtype: int64

Region
Region
Uttar Pradesh     96
Madhya Pradesh    95
Gujarat           86
Maharashtra       84
Odisha            83
Tamil Nadu        83
Rajasthan         82
Bihar             81
West Bengal       80
Andhra Pradesh    79
Haryana           76
Punjab            75
Name: count, dtype: int64

Material_Availability
Material_Availability
Medium    456
High      290
Low       254
Name: count, dtype: int64

Equipment_Availability
Equipment_Availability
Medium    475
High      323
Low       202
Name: count, dtype: int64

Weather_Risk
Weather_Risk
Low       447
Medium    336
High      217
Name: count, dtype: int64

Supplier_Delay
Supplier_Delay
No     717
Yes    283
Name: count, dtype: int64

Project_Complexity
Project_Complexity
Medium    460
Low       343
High      197
Name: count, dtype: int64

Delay_

In [14]:
# Separate the target from the input features.
target = "Delay_Risk"

X = df.drop(columns=["Project_ID", target])
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())

Feature shape: (1000, 13)
Target shape: (1000,)


,Project_Type,Region,Planned_Duration_Months,Project_Progress_Percent,Budget_Million_DZD,Cost_Used_Percent,Number_of_Workers,Material_Availability,Equipment_Availability,Weather_Risk,Supplier_Delay,Project_Complexity,Safety_Incidents
0,Building,Madhya Pradesh,17,68,4684.52,94.17,44,Medium,High,Medium,No,Medium,1
1,Industrial Site,Maharashtra,47,95,2732.85,23.61,757,Low,High,Low,No,Medium,1
2,Housing,Punjab,43,55,4701.27,129.63,76,Medium,Medium,High,No,Medium,1
3,Pipeline,Maharashtra,23,54,1141.39,32.01,199,Medium,High,High,Yes,High,1
4,Road,West Bengal,40,10,3642.34,81.30,251,Low,High,Low,No,Medium,1


In [15]:
## Split the data before preprocessing.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 800
Testing rows: 200


In [16]:
## Imputation: fills missing values if they appear in future versions of the dataset.
## Scaling: puts numerical features on comparable scales.
## One-hot encoding: converts labels such as Building, Road, and Bridge into numerical features.
## Pipeline: fits preprocessing on training data and applies the same transformations to testing data.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_columns = X_train.select_dtypes(
    include=["number"]).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include=["object", "category", "str"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)])

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

print("Training data prepared:", X_train_ready.shape)
print("Testing data prepared:", X_test_ready.shape)

## Verify the prepared data.
print("Training samples:", X_train_ready.shape[0])
print("Testing samples:", X_test_ready.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training data prepared: (800, 38)
Testing data prepared: (200, 38)
Training samples: 800
Testing samples: 200

Training target distribution:
Delay_Risk
No     478
Yes    322
Name: count, dtype: int64

Testing target distribution:
Delay_Risk
No     119
Yes     81
Name: count, dtype: int64


In [17]:
from pathlib import Path
import joblib

OUTPUT_DIR = Path(r"C:\Users\Soumya_SRB\Desktop\L&T Project Intelligencee\Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the cleaned dataset
df.to_csv(OUTPUT_DIR /"cleaned_data.csv", index=False)

# Save the exact train/test split and fitted preprocessor
joblib.dump(
    {"X_train": X_train,
     "X_test": X_test,
     "y_train": y_train,
     "y_test": y_test,
     "preprocessor": preprocessor},
      OUTPUT_DIR / "prepared_data.joblib")

print("Prepared data saved to:", OUTPUT_DIR)
print("File exists:", (OUTPUT_DIR / "prepared_data.joblib").exists())

Prepared data saved to: C:\Users\Soumya_SRB\Desktop\L&T Project Intelligencee\Outputs
File exists: True
